In [1]:
from pathlib import Path
import random
import sys
from datasets import load_dataset
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
# from datasets import load_dataset

REQUESTED_DEVICE = "cpu"  # Options: "auto", "cpu", "cuda", "cuda:0", "mps", ...


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "trainable_sae.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from trainable_sae import (
    HookPointSpec,
    SAEConfig,
    SAEConnector,
    TrainableSAE,
    load_hooked_transformer,
    resolve_device,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = resolve_device(REQUESTED_DEVICE)
dtype = torch.float32
device, dtype

d:\andyh\Documents\Projects\mines\MATH498-Sparse-Autoencoder-Manipulation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True
12.6


('cpu', torch.float32)

In [2]:
import time
MODEL_NAME = "google/gemma-3-270m-it"

PILE_DATASET = "HuggingFaceFW/fineweb"
PILE_DATASET_CONFIG = "sample-10BT"

QUESTIONS_PATH = PROJECT_ROOT / "samples/questions_train.txt"

PILE_TOKEN_BUDGET = 50_000_000
QUESTION_REPEATS = 3
BATCH_TOKENS = 8192
MODEL_FORWARD_TEXTS = 128
CONTEXT_SIZE = 64
EXPANSION_FACTOR = 32
TOP_K = 50
LR = 1e-4
MIN_LR = 5e-6
WARMUP_STEPS = 100
MAX_STEPS = None
SCHEDULER_TOTAL_STEPS = MAX_STEPS or max(1, PILE_TOKEN_BUDGET // BATCH_TOKENS)

SAVE_DIR = PROJECT_ROOT / f"custom_saes/gemma3_270m_mid_resid_post_shrink_fineweb50m_questions{int(time.time())}"


In [3]:
model = load_hooked_transformer(
    MODEL_NAME,
    device=device,
    dtype=dtype,
)

mid_layer = model.cfg.n_layers // 2
hook_point = HookPointSpec(layer=mid_layer, site="resid_post").name
d_in = model.cfg.d_model

print(f"model:      {MODEL_NAME}")
print(f"layers:     {model.cfg.n_layers}")
print(f"hook point: {hook_point}")
print(f"d_in:       {d_in}")

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 5316.60it/s]


Loaded pretrained model google/gemma-3-270m-it into HookedTransformer
Moving model to device:  cpu
model:      google/gemma-3-270m-it
layers:     18
hook point: blocks.9.hook_resid_post
d_in:       640
